In [20]:
import pandas as pd
import pandas_gbq
from sqlalchemy import create_engine
import urllib.parse
import pymssql
from datetime import datetime
import numpy as np
from google.cloud import bigquery
import numpy as np

In [ ]:
caminho = r'C:\Users\Nelio\Documents\GitHub\airflow_nelio\notebooks\RelatorioDeContribuintes (2).txt'
dados = []

with open(caminho, 'r', encoding='latin1') as f:
    linhas = f.readlines()
    cabecalho = linhas[0].strip().split(';')
    
    for i, linha in enumerate(linhas[1:]): # Pula o cabeçalho
        colunas = linha.strip().split(';')
        
        # Se a linha tiver mais colunas que o cabeçalho, 
        # juntamos o excesso na última coluna (AtividadesSecundarias)
        if len(colunas) > 4:
            principal = colunas[:3]
            resto = ", ".join(colunas[3:]) # Junta tudo que sobrou
            dados.append(principal + [resto])
        else:
            # Garante que a linha tenha exatamente 4 elementos
            while len(colunas) < 4:
                colunas.append("")
            dados.append(colunas[:4])

df = pd.DataFrame(dados, columns=['CodigoContribuinte', 'NomeContribuinte', 'AtividadePrimaria', 'AtividadesSecundarias'])

print(f"Total de linhas processadas: {len(df)}")

Total de linhas processadas: 38626


In [12]:
df = df[['CodigoContribuinte', 'AtividadePrimaria', 'AtividadesSecundarias']]

In [27]:
df_trabalho = df.copy()

# Combina primária e secundárias em uma única lista por linha.
df_trabalho['AtividadePrimaria'] = df_trabalho['AtividadePrimaria'].fillna('').astype(str)
df_trabalho['AtividadesSecundarias'] = df_trabalho['AtividadesSecundarias'].fillna('').astype(str)

df_trabalho['Atividade Economica'] = (
    df_trabalho['AtividadePrimaria'].str.split(',')
    + df_trabalho['AtividadesSecundarias'].str.split(',')
)

df_trabalho['Atividade Economica'] = df_trabalho['Atividade Economica'].apply(
    lambda lista: [item.strip() for item in lista if item and item.strip()]
)

df_long = df_trabalho[['CodigoContribuinte', 'Atividade Economica']].explode('Atividade Economica')
df_long = df_long.dropna(subset=['Atividade Economica'])

df_consolidado = df_long.groupby('CodigoContribuinte', as_index=False)['Atividade Economica'].agg(
    lambda serie: list(dict.fromkeys(serie))
)

print(f"Linhas originais: {len(df)}")
print(f"Contribuintes únicos: {df['CodigoContribuinte'].nunique()}")
print(f"Linhas consolidadas: {len(df_consolidado)}")

df_consolidado

Linhas originais: 38626
Contribuintes únicos: 38577
Linhas consolidadas: 38577


,CodigoContribuinte,Atividade Economica
0,1000003,[56112.01 - Restaurantes e similares]
1,1000034,"[64221.00 - Bancos múltiplos, com carteira com..."
2,1000049,[23427.02 - Fabricação de artefatos de cerâmic...
3,1000054,[47890.99 - Comércio varejista de outros produ...
4,1000058,"[47822.01 - Comércio varejista de calçados, 47..."
...,...,...
38572,999810649,[. -]
38573,999810838,[. -]
38574,999814639,[. -]
38575,9999619,[. -]


In [28]:
df_consolidado['Atividade Economica'] = df_consolidado['Atividade Economica'].apply(
    lambda atividades: ', '.join(atividades) if isinstance(atividades, list) else str(atividades)
)

df_consolidado

,CodigoContribuinte,Atividade Economica
0,1000003,56112.01 - Restaurantes e similares
1,1000034,"64221.00 - Bancos múltiplos, com carteira come..."
2,1000049,23427.02 - Fabricação de artefatos de cerâmica...
3,1000054,47890.99 - Comércio varejista de outros produt...
4,1000058,"47822.01 - Comércio varejista de calçados, 475..."
...,...,...
38572,999810649,. -
38573,999810838,. -
38574,999814639,. -
38575,9999619,. -


In [22]:
usuario = 'ib_dw'
senha = urllib.parse.quote_plus('Ib160275@') 
host = '186.248.197.68'
porta = '5000'
database = 'gissonline'
conn_str = f"mssql+pymssql://{usuario}:{senha}@{host}:{porta}/{database}?charset=utf8"
engine = create_engine(conn_str)
print("Engine pymssql configurado!")

queryita = """

WITH regim AS (
    SELECT 
        A.[CodigoContribuinte],
        A.[NumeroProcesso],
        A.[DtEntrada],
        ROW_NUMBER() OVER (
            PARTITION BY A.[CodigoContribuinte] 
            ORDER BY A.[DtEntrada] DESC
        ) AS Rnk,
        A.[Estagio],
        A.[Regin],
        B.CodigoAtividade,
        B.ComplAtividade,
        B.TipoAtividade
    FROM [ib_iss].[dbo].[Protocolo] AS A
    LEFT JOIN [ib_iss].[dbo].[Contribu] AS B 
        ON A.[CodigoContribuinte] = B.[CodigoContribuinte]
    --WHERE A.CodigoContribuinte = '1034767'
),

EMPRESAS AS (
    SELECT  
        NUM_CADASTRO,
        NOME_EMPRESA,
        INSCRICAO,
        RIGHT(REPLICATE('0', 14) + CPF_CNPJ, 14) AS CNPJ_CONVERTIDO,
        TIPO_EMPRESA,
        BAIRRO,
        DATA_ABERTURA,
        DATA_ENCERRAMENTO,
        STATUS_EMPRESA
    FROM [gissonline].[dbo].TB_INTER_EMPRESAS
),

SUSPENSA AS (
    SELECT * FROM [gissonline].[dbo].TB_INTER_EMPR_SUSPENSA
    WHERE DATA_SUSPENSAO_FIM IS NULL
),

SIMPLES AS (
    SELECT 
        *,
        'S' AS OPCAO_PELO_SIMPLES
    FROM [gissonline].[dbo].TB_INTER_EMPR_SNACIONAL
    WHERE DATA_FIM IS NULL 
      AND DATA_INICIO IS NOT NULL
),

MEI AS (
    SELECT 
        *,
        'S' AS OPCAO_PELO_MEI
    FROM [gissonline].[dbo].TB_INTER_EMPR_MEI
    WHERE DATA_FIM IS NULL 
      AND DATA_INICIO IS NOT NULL
),

fechado AS (
    SELECT
        CAST(EMPRESAS.NUM_CADASTRO AS VARCHAR) AS INSCRICAO,
        EMPRESAS.NOME_EMPRESA,
        EMPRESAS.TIPO_EMPRESA,
        EMPRESAS.CNPJ_CONVERTIDO AS CNPJ,
        'ATIVIDADE_ECONOMICA' AS ATIVIDADE_ECONOMICA,
        EMPRESAS.STATUS_EMPRESA AS SITUACAO_CADASTRAL,
        CASE  
            WHEN MEI.OPCAO_PELO_MEI = 'S' THEN 'MEI'
            WHEN SIMPLES.OPCAO_PELO_SIMPLES = 'S' THEN 'SIMPLES'
            ELSE 'NORMAL'
        END AS REGIME_TRIBUTARIO,
        regim.DtEntrada AS DtAtualizacao,
        regim.CodigoAtividade,
        regim.ComplAtividade,
        regim.TipoAtividade,
        regim.Rnk
    FROM EMPRESAS
    LEFT JOIN SUSPENSA 
        ON SUSPENSA.NUM_CADASTRO = EMPRESAS.NUM_CADASTRO
    LEFT JOIN SIMPLES 
        ON SIMPLES.NUM_CADASTRO = EMPRESAS.NUM_CADASTRO
    LEFT JOIN MEI 
        ON MEI.NUM_CADASTRO = EMPRESAS.NUM_CADASTRO
    LEFT JOIN regim 
        ON EMPRESAS.INSCRICAO = regim.CodigoContribuinte
)

SELECT * FROM fechado
WHERE Rnk = 1
  --AND CNPJ = '33643150000172'
  order by DtAtualizacao desc
"""
ita = pd.read_sql(queryita, engine)


print(f"Total de linhas: {len(ita)}")
ita

Engine pymssql configurado!
Total de linhas: 49726


,INSCRICAO,NOME_EMPRESA,TIPO_EMPRESA,CNPJ,ATIVIDADE_ECONOMICA,SITUACAO_CADASTRAL,REGIME_TRIBUTARIO,DtAtualizacao,CodigoAtividade,ComplAtividade,TipoAtividade,Rnk
0,1036162,A V S DE ITABORAI MERCEARIA EIRELI ME,J,20402775000109,ATIVIDADE_ECONOMICA,A,NORMAL,3015-03-30 00:00:00,None,None,None,1
1,1038002,ARGAMASSA CARIOCA LTDA,J,11335362000150,ATIVIDADE_ECONOMICA,A,NORMAL,2209-10-01 00:00:00,None,None,None,1
2,7009327,40.183.243 JESSICA CRISTINA MAIA ROSA,J,40183243000127,ATIVIDADE_ECONOMICA,A,MEI,2202-03-01 03:00:00,None,None,None,1
3,7006695,CLOVIS HARDOIM LIRIO,J,36238733000106,ATIVIDADE_ECONOMICA,E,MEI,2202-01-25 01:00:00,None,None,None,1
4,7026011,57.098.206 JOAO VITOR SILVA DO NASCIMENTO,J,57098206000113,ATIVIDADE_ECONOMICA,E,MEI,2055-05-29 00:00:00,None,None,None,1
...,...,...,...,...,...,...,...,...,...,...,...,...
49721,1029774,INSTITUTO SOCIAL ARAUJO,J,05418471000219,ATIVIDADE_ECONOMICA,E,NORMAL,None,None,None,None,1
49722,1029986,TANIA MARIA SIQUEIRA BITTENCOURT,F,00077850599772,ATIVIDADE_ECONOMICA,E,NORMAL,None,None,ARTIGOS PRESENTES E BIJUTERIAS,None,1
49723,7000001,L.L. SILVA DA NETO ELETRONS INSTALAÇÕES,J,10989878000155,ATIVIDADE_ECONOMICA,E,NORMAL,None,None,None,None,1
49724,2000006,SERGIO DA SILVA GUILHERME,F,00000001908707,ATIVIDADE_ECONOMICA,A,NORMAL,None,None,None,None,1


In [ ]:
empre_cnae = pd.merge(
    ita, 
    df_consolidado, 
    left_on='INSCRICAO', 
    right_on='CodigoContribuinte', how='left')
empre_cnae = empre_cnae[[ 
                'INSCRICAO',
                'NOME_EMPRESA', 
                'TIPO_EMPRESA',
                'CNPJ',
                'Atividade Economica',
                'SITUACAO_CADASTRAL',
                'REGIME_TRIBUTARIO',
                'DtAtualizacao'
            ]]


In [31]:
empre_cnae.to_csv(r'C:\Users\Nelio\Documents\GitHub\airflow_nelio\notebooks\empre_cnae.csv', index=False, encoding='utf-8-sig')

In [33]:
empre_cnae

,INSCRICAO,NOME_EMPRESA,TIPO_EMPRESA,CNPJ,Atividade Economica,SITUACAO_CADASTRAL,REGIME_TRIBUTARIO,DtAtualizacao
0,1036162,A V S DE ITABORAI MERCEARIA EIRELI ME,J,20402775000109,47121.00 - Comércio varejista de mercadorias e...,A,NORMAL,3015-03-30 00:00:00
1,1038002,ARGAMASSA CARIOCA LTDA,J,11335362000150,23303.05 - Preparação de massa de concreto e a...,A,NORMAL,2209-10-01 00:00:00
2,7009327,40.183.243 JESSICA CRISTINA MAIA ROSA,J,40183243000127,87123.00 - Atividades de fornecimento de infra...,A,MEI,2202-03-01 03:00:00
3,7006695,CLOVIS HARDOIM LIRIO,J,36238733000106,NaN,E,MEI,2202-01-25 01:00:00
4,7026011,57.098.206 JOAO VITOR SILVA DO NASCIMENTO,J,57098206000113,NaN,E,MEI,2055-05-29 00:00:00
...,...,...,...,...,...,...,...,...
49721,1029774,INSTITUTO SOCIAL ARAUJO,J,05418471000219,NaN,E,NORMAL,None
49722,1029986,TANIA MARIA SIQUEIRA BITTENCOURT,F,00077850599772,NaN,E,NORMAL,None
49723,7000001,L.L. SILVA DA NETO ELETRONS INSTALAÇÕES,J,10989878000155,NaN,E,NORMAL,None
49724,2000006,SERGIO DA SILVA GUILHERME,F,00000001908707,. -,A,NORMAL,None


In [32]:
df

,CodigoContribuinte,AtividadePrimaria,AtividadesSecundarias
0,7025400,22293.99 - Fabricação de artefatos de material...,","
1,7025467,. -,","
2,7026079,47814.00 - Comércio varejista de artigos do ve...,43223.02 - Instalação e manutenção de sistemas...
3,7025463,85996.99 - Outras atividades de ensino não esp...,","
4,5041977,00000.N - ATIVIDADE GERAL,","
...,...,...,...
38621,6521680,. -,","
38622,7001906,82300.02 - Casas de festas e eventos,47547.01 - Comércio varejista de móveis 47814....
38623,7008032,97005.00 - Serviços domésticos,","
38624,1026470,00000.N - ATIVIDADE GERAL,","
